# Step 3 — Model Definition

We build a lightweight **ResNet-style CNN** that maps 5-band galaxy images to 4 morphological parameters.

## Design choices

| Decision | Choice | Reason |
|----------|--------|--------|
| Input channels | 5 | One per HSC band (g, r, i, z, y) |
| Spatial size | 64×64 | Fixed by the dataset |
| Backbone | 5× (Conv stride-2 + ResBlock) | Halves spatial dims each stage; residual connections help gradients flow |
| Normalisation | BatchNorm | Stable training without large batch sizes |
| Pooling | Global Average Pool | Removes spatial dims without a hard-coded FC size |
| Head | FC 256→128→4 + Dropout 0.3 | Light MLP; dropout reduces overfitting on the small 50 K subset |
| Loss | Huber (δ=1) | Less sensitive to outliers than MSE, smoother than MAE |
| Outputs | 4 (normalised) | Targets are log/standardised — predicted in normalised space, inverted at eval |

## 3.1 Import & Instantiate

In [6]:
import os, sys
import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath("step3_model.ipynb"))
sys.path.insert(0, HERE)
from model import MorphCNN

model = MorphCNN(n_outputs=4)
print(model)

MorphCNN(
  (encoder): Sequential(
    (0): Sequential(
      (0): Conv2d(5, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): ResBlock(
        (block): Sequential(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
          (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (relu): ReLU(inplace=True)
      )
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU

## 3.2 Parameter Count per Layer

In [7]:
print(f"{'Module':<45} {'Params':>10}")
print("-" * 57)
total = 0
for name, module in model.named_modules():
    own_params = sum(p.numel() for p in module.parameters(recurse=False) if p.requires_grad)
    if own_params > 0:
        print(f"  {name:<43} {own_params:>10,}")
        total += own_params
print("-" * 57)
print(f"  {'TOTAL':<43} {total:>10,}")

Module                                            Params
---------------------------------------------------------
  encoder.0.0                                      1,440
  encoder.0.1                                         64
  encoder.0.3.block.0                              9,216
  encoder.0.3.block.1                                 64
  encoder.0.3.block.3                              9,216
  encoder.0.3.block.4                                 64
  encoder.1.0                                     18,432
  encoder.1.1                                        128
  encoder.1.3.block.0                             36,864
  encoder.1.3.block.1                                128
  encoder.1.3.block.3                             36,864
  encoder.1.3.block.4                                128
  encoder.2.0                                     73,728
  encoder.2.1                                        256
  encoder.2.3.block.0                            147,456
  encoder.2.3.block.1         

## 3.3 Feature Map Sizes Through the Network

Trace a dummy `(1, 5, 64, 64)` tensor through the encoder to see the spatial dimensions shrink at each stage.

In [8]:
model.eval()
x = torch.zeros(1, 5, 64, 64)

stage_names = [
    "Input",
    "Stage 1  (5→32  ch)",
    "Stage 2  (32→64  ch)",
    "Stage 3  (64→128 ch)",
    "Stage 4  (128→256 ch)",
    "Stage 5  (256→256 ch)",
    "Global Avg Pool",
    "Head output",
]

shapes = [tuple(x.shape)]
with torch.no_grad():
    for stage in model.encoder:
        x = stage(x)
        shapes.append(tuple(x.shape))
    x = model.pool(x)
    shapes.append(tuple(x.shape))
    x = model.head(x)
    shapes.append(tuple(x.shape))

print(f"{'Stage':<30} {'Shape'}")
print("-" * 50)
for name, shape in zip(stage_names, shapes):
    print(f"  {name:<28} {str(shape)}")

Stage                          Shape
--------------------------------------------------
  Input                        (1, 5, 64, 64)
  Stage 1  (5→32  ch)          (1, 32, 32, 32)
  Stage 2  (32→64  ch)         (1, 64, 16, 16)
  Stage 3  (64→128 ch)         (1, 128, 8, 8)
  Stage 4  (128→256 ch)        (1, 256, 4, 4)
  Stage 5  (256→256 ch)        (1, 256, 2, 2)
  Global Avg Pool              (1, 256, 1, 1)
  Head output                  (1, 4)


## 3.4 Architecture Diagram

A schematic of the full network drawn with matplotlib — useful for papers / presentations.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(16, 3.2))
ax.set_xlim(0, 15.5)
ax.set_ylim(0, 4)
ax.axis("off")
fig.patch.set_facecolor("#f8f9fa")

blocks = [
    # (x_centre, label_top, label_bot, width, color)
    (0.75, "Input",              "5×64×64",             0.75, "#aec6cf"),
    (2.2,  "Stage 1",            "32×32×32\n+ResBlock",  0.85, "#b5ead7"),
    (3.7,  "Stage 2",            "64×16×16\n+ResBlock",  0.85, "#b5ead7"),
    (5.2,  "Stage 3",            "128×8×8\n+ResBlock",   0.85, "#b5ead7"),
    (6.7,  "Stage 4",            "256×4×4\n+ResBlock",   0.85, "#b5ead7"),
    (8.2,  "Stage 5",            "256×2×2\n+ResBlock",   0.85, "#b5ead7"),
    (9.85, "Global\nAvg Pool",   "256×1×1",             0.90, "#ffd700"),
    (11.5, "FC Head",            "256→128→4",           0.85, "#ffb347"),
    (13.0, "Output",             "4 params",            0.75, "#ff6b6b"),
]

for (xc, top, bot, w, col) in blocks:
    rect = mpatches.FancyBboxPatch(
        (xc - w/2, 0.75), w, 2.2,
        boxstyle="round,pad=0.07",
        facecolor=col, edgecolor="#555", linewidth=1.1
    )
    ax.add_patch(rect)
    ax.text(xc, 3.05, top, ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.text(xc, 1.85, bot, ha="center", va="center", fontsize=7,
            color="#333", linespacing=1.4)

# arrows
for i in range(len(blocks) - 1):
    xc_l, _, _, w_l, _ = blocks[i]
    xc_r, _, _, w_r, _ = blocks[i + 1]
    ax.annotate("", xy=(xc_r - w_r/2, 1.85),
                xytext=(xc_l + w_l/2, 1.85),
                arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.4))

# legend
inp_patch = mpatches.Patch(color="#aec6cf", label="Input image")
res_patch = mpatches.Patch(color="#b5ead7", label="Conv(stride=2) + BN + ReLU + ResBlock")
gap_patch = mpatches.Patch(color="#ffd700", label="Global Average Pooling")
fc_patch  = mpatches.Patch(color="#ffb347", label="FC + Dropout(0.3)")
out_patch = mpatches.Patch(color="#ff6b6b", label="4 morphology params")
ax.legend(handles=[inp_patch, res_patch, gap_patch, fc_patch, out_patch],
          loc="lower center", ncol=5, fontsize=7.5, framealpha=0.8,
          bbox_to_anchor=(0.5, -0.08))

ax.set_title("MorphCNN Architecture", fontsize=12, fontweight="bold", pad=6)
plt.tight_layout()
plt.savefig(os.path.join(HERE, "figures/architecture.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved architecture.png")

## 3.5 Receptive Field & Forward Pass Sanity Check

Verify the model runs without errors and the output shape is correct.

In [10]:
batch = torch.randn(8, 5, 64, 64)   # typical mini-batch
with torch.no_grad():
    out = model(batch)

print(f"Input shape:   {tuple(batch.shape)}")
print(f"Output shape:  {tuple(out.shape)}   (batch=8, 4 parameters)")
print(f"Output range:  [{out.min():.3f}, {out.max():.3f}]  (normalised space — expected ~[-3, 3])")
print()
param_names = ["ellipticity", "major_axis", "sersic_index", "isophotal_area"]
for i, name in enumerate(param_names):
    print(f"  {name:<18}  pred mean={out[:,i].mean():.3f}  std={out[:,i].std():.3f}")

Input shape:   (8, 5, 64, 64)
Output shape:  (8, 4)   (batch=8, 4 parameters)
Output range:  [-0.091, 0.050]  (normalised space — expected ~[-3, 3])

  ellipticity         pred mean=0.012  std=0.000
  major_axis          pred mean=-0.090  std=0.001
  sersic_index        pred mean=0.011  std=0.001
  isophotal_area      pred mean=0.050  std=0.000
